# Microstructure Signal Lab

Notebook for testing a simple short-horizon microstructure signal on replayed futures-like data. Goal is to demonstrate a research workflow similar to those in a market-making or risk-monitoring system.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px

root = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import ReplayConfig
from src.replay import MarketReplay
from src.microstructure import add_microstructure_features

## Data

Will be generating deterministic replay data to develop a working baseline for now; should mirror live data formats to enable future transition. The data includes top-of-book bid/ask, size, mid, spread, imbalance, and latency proxies.

In [2]:
cfg = ReplayConfig(n_events=3000)
raw = MarketReplay(cfg).generate()
raw.head()

,ts,bid,ask,mid,bid_size,ask_size,trade_px,latency_ms,spread,spread_bps,imbalance,mid_ret
0,2026-01-01 09:30:00,5000.25,5000.75,5000.5,27,4,5000.5,92.275017,0.50,0.9999,0.741935,0.0000
1,2026-01-01 09:30:01,4998.25,4998.75,4998.5,26,10,4998.5,81.085611,0.50,1.0003,0.444444,-0.0004
2,2026-01-01 09:30:02,4999.50,5000.50,5000.0,16,19,5000.0,27.183184,1.00,2.0000,-0.085714,0.0003
3,2026-01-01 09:30:03,5001.75,5002.00,5002.0,31,3,5002.0,87.026942,0.25,0.4998,0.823529,0.0004
4,2026-01-01 09:30:04,4997.75,4998.25,4998.0,29,14,4998.0,86.988404,0.50,1.0004,0.348837,-0.0008


## Features

Basic features: spread, imbalance, EWMA variance, a z-score of short-horizon returns, and a simple signal that mean-reverts the z-score.

Rolling statistics are shifted by one observation so each signal only uses information that would have been known before the current decision point.

Can adjust signal parameters such as z-score threshold for entry here.

In [3]:
feat = add_microstructure_features(raw, window=30, entry_z = 2.3)
feat[['ts', 'mid', 'spread_bps', 'imbalance', 'ewma_var', 'signal']].head(5)

,ts,mid,spread_bps,imbalance,ewma_var,signal
0,2026-01-01 09:30:00,5000.5,0.9999,0.741935,0.000000e+00,0
1,2026-01-01 09:30:01,4998.5,1.0003,0.444444,7.998400e-08,0
2,2026-01-01 09:30:02,5000.0,2.0000,-0.085714,6.822309e-08,0
3,2026-01-01 09:30:03,5002.0,0.4998,0.823529,7.633393e-08,0
4,2026-01-01 09:30:04,4998.0,1.0004,0.348837,1.587412e-07,0


## Signal test

Evaluating: if the signal is strong enough, treat it as a directional forecast for the next bar. Meant as a methodology demo.

hit_rate: Percentage of signal events where predicted direction matches the next bar's direction.\
avg_fwd_ret: Average direction-adjusted return of the next bar after a signal fires.\
coverage: Percentage of rows where signal was non-zero. Higher coverage = more trades.



In [4]:
df = feat.copy()

df["fwd_ret"] = df["mid"].pct_change().shift(-1).fillna(0.0)
df["pred"] = np.sign(df["signal"])
df["actual"] = np.sign(df["fwd_ret"])
df["correct"] = (df["pred"] == df["actual"]) & (df["pred"] != 0)
df["directional_fwd_ret"] = df["signal"] * df["fwd_ret"]

active = df[df["pred"] != 0]

hit_rate = active["correct"].mean()
avg_fwd_ret = active["directional_fwd_ret"].mean()
coverage = (df["pred"] != 0).mean()

results = pd.DataFrame({
    "metric": ["hit_rate", "avg_fwd_ret", "coverage"],
    "value": [hit_rate, avg_fwd_ret, coverage]
})

results["value"] = results.apply(
    lambda row: f"{row['value']:.2%}" if row["metric"] in ["hit_rate", "coverage"]
    else f"{row['value']:.6f}",
    axis=1
)

results

,metric,value
0,hit_rate,46.23%
1,avg_fwd_ret,-0.000055
2,coverage,3.53%


## Visualization

Mid-price and Z-score signal charts

In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "vscode"

plot_df = df.tail(500).copy()
entry_df = plot_df.loc[plot_df["signal"] != 0].copy()

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Mid Price", "Z-Score and Signal Triggers")
)

fig.add_trace(
    go.Scatter(
        x=plot_df["ts"],
        y=plot_df["mid"],
        name="Mid",
        mode="lines",
        line=dict(color="#1f77b4"),
        showlegend=True,
        legendgroup="top"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=plot_df["ts"],
        y=plot_df["zscore"],
        name="Z-Score",
        mode="lines",
        line=dict(dash="dot", color="gray"),
        showlegend=True,
        legendgroup="bottom"
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=entry_df["ts"],
        y=entry_df["zscore"],
        mode="markers",
        name="Signal Trigger",
        marker=dict(
            size=10,
            color=entry_df["signal"],
            colorscale="RdBu",
            showscale=False,
            symbol="circle"
        ),
        showlegend=True,
        legendgroup="bottom"
    ),
    row=2, col=1
)

fig.update_layout(
    height=700,
    title="Mid Price and Mean-Reversion Signal",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=.57,
        xanchor="left",
        x=1.02
    ),
    margin=dict(r=160)
)

fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Z-Score", row=2, col=1)
fig.show()

trade_log = entry_df[["ts", "signal", "mid", "trade_px", "zscore", "signal_strength"]].copy()
trade_log["direction"] = trade_log["signal"].map({1: "Long", -1: "Short"})
trade_log["mid"] = trade_log["mid"].round(2)
trade_log["trade_px"] = trade_log["trade_px"].round(2)
trade_log["zscore"] = trade_log["zscore"].round(2)
trade_log["signal_strength"] = trade_log["signal_strength"].round(2)
trade_log = trade_log[["ts", "direction", "trade_px", "mid", "zscore", "signal_strength"]]

trade_log.head(5)

,ts,direction,trade_px,mid,zscore,signal_strength
2519,2026-01-01 10:11:59,Long,4796.00,4796.00,-2.41,2.41
2649,2026-01-01 10:14:09,Short,4834.75,4834.75,2.56,2.56
2651,2026-01-01 10:14:11,Long,4828.75,4828.75,-2.39,2.39
2707,2026-01-01 10:15:07,Long,4816.50,4816.50,-2.89,2.89
2738,2026-01-01 10:15:38,Short,4812.25,4812.25,3.09,3.09


## Discussion

This notebook is designed to show a research workflow: build features, test a hypothesis, inspect results, and think about deployment constraints. Limitations include synthetic data, no depth-of-book information, and a simple next-bar direction label.

Hope to use this framework with real data and more complex strategies moving forward.